# 04. Load to PostgreSQL

**Цель:** прочитать все витрины и записать их в PostgreSQL для дальнейшего подключения Power BI.

## 0. Импорт и инициализация

In [1]:
import sys
sys.path.append("..")

from utils.spark_session import get_spark_session
from utils.config import AGGREGATES_DIR, JDBC_URL, JDBC_PROPERTIES

spark = get_spark_session()

## 1. Запись

In [4]:
mart_names = [
    "funnel_daily",
    "category_sales",
    "user_metrics",
    "cart_abandonment",
    "time_patterns",
    "daily_summary",
]

for mart_name in mart_names:
    parquet_path = f"{AGGREGATES_DIR}/{mart_name}.parquet"
    df = spark.read.parquet(parquet_path)

    (
        df.write
        .jdbc(url=JDBC_URL, table=mart_name, mode="overwrite", properties=JDBC_PROPERTIES)
    )

    print(f"{mart_name}: {df.count():,} строк")

funnel_daily: 93 строк
category_sales: 2,885 строк
user_metrics: 3,022,290 строк
cart_abandonment: 797 строк
time_patterns: 504 строк
daily_summary: 31 строк


## 2. Проверка загрузки

Сверяем количество строк в Postgres с тем, что было в Parquet до записи.

In [3]:
for mart_name in mart_names:
    parquet_path = f"{AGGREGATES_DIR}/{mart_name}.parquet"

    parquet_count = spark.read.parquet(parquet_path).count()
    postgres_count = spark.read.jdbc(url=JDBC_URL, table=mart_name, properties=JDBC_PROPERTIES).count()

    status = "OK" if parquet_count == postgres_count else "MISMATCH"
    print(f"{mart_name:<20} parquet: {parquet_count:>10,}  postgres: {postgres_count:>10,}  [{status}]")

funnel_daily         parquet:         93  postgres:         93  [OK]
category_sales       parquet:      2,885  postgres:      2,885  [OK]
user_metrics         parquet:  3,022,290  postgres:  3,022,290  [OK]
cart_abandonment     parquet:        797  postgres:        797  [OK]
time_patterns        parquet:        504  postgres:        504  [OK]
daily_summary        parquet:         31  postgres:         31  [OK]


## Итог

Все агрегированные витрины успешно загружены в PostgreSQL и готовы к использованию в PowerBI.